# Movie Review Sentiment Analysis Using NLP and Machine Learning

## Project aim
Build a machine-learning system that reads a movie review and classifies its sentiment as **Positive** or **Negative**.

This notebook contains:
- Project theory and important concepts
- Dataset loading and exploration
- Text cleaning and preprocessing
- TF-IDF feature extraction
- Logistic Regression model training
- Evaluation using accuracy, precision, recall, F1-score, and a confusion matrix
- Prediction on new reviews
- Optional comparison with Naive Bayes and Linear SVM

**Dataset:** [IMDB Dataset of 50K Movie Reviews](https://ai.stanford.edu/~amaas/data/sentiment/).  
For the CSV-based workflow below, download `IMDB Dataset.csv` and upload it to the notebook's working directory (or Google Colab session storage). It should contain columns named `review` and `sentiment`.


## 1. Introduction and theory

**Sentiment analysis** is the task of identifying the opinion or emotion expressed in text. In this project, it is a binary classification problem:
- `positive` → label `1`
- `negative` → label `0`

**Natural Language Processing (NLP)** is a field of artificial intelligence that enables computers to process and analyse human language. Movie reviews are unstructured text, so we must transform them into numerical features before using most traditional machine-learning algorithms.

### Project workflow
1. Load labelled movie reviews.
2. Explore the dataset and check for missing or duplicate records.
3. Clean the text while preserving sentiment-bearing words such as *not* and *never*.
4. Split the data into training and test sets.
5. Convert text into TF-IDF features.
6. Train a supervised classification model.
7. Evaluate performance on unseen test data.
8. Use the trained pipeline to classify new reviews.


## 2. Important theoretical concepts

### 2.1 Text preprocessing
Preprocessing makes text more consistent and removes irrelevant formatting. Common steps include:
- **Lowercasing:** makes `GREAT` and `great` consistent.
- **HTML removal:** removes tags such as `<br />`.
- **Punctuation/number cleanup:** removes characters that are not useful for this experiment.
- **Whitespace cleanup:** removes repeated spaces.

Tokenization splits text into tokens (usually words). Stopword removal and stemming/lemmatization are optional. In this notebook, we do not remove stopwords because words such as **not** can reverse sentiment: “good” and “not good” mean different things.

### 2.2 TF-IDF
Machine-learning models need numeric input. **Term Frequency–Inverse Document Frequency (TF-IDF)** represents each review as a vector of word or phrase weights.

\[
TFIDF(t,d)=TF(t,d)\times IDF(t)
\]

- **TF:** how often a term occurs in a document.
- **IDF:** reduces the weight of terms that occur in many documents.
- **Unigrams:** single words, e.g. `excellent`.
- **Bigrams:** two-word phrases, e.g. `not good`.

The vectorizer is fitted only on training data to avoid information leakage from the test set.

### 2.3 Logistic Regression
Logistic Regression is a supervised learning algorithm commonly used for binary classification. It estimates the probability that a review belongs to the positive class.

\[
P(y=1\mid x)=\frac{1}{1+e^{-z}},\qquad z=w^Tx+b
\]

If the probability is at least 0.5, the review is classified as positive; otherwise, it is classified as negative. The model learns weights for the TF-IDF features during training.

### 2.4 Evaluation metrics
- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** among reviews predicted positive, the fraction that truly are positive.
- **Recall:** among all truly positive reviews, the fraction found by the model.
- **F1-score:** harmonic mean of precision and recall.
- **Confusion matrix:** counts true negatives, false positives, false negatives, and true positives.

Accuracy alone may be misleading on imbalanced datasets, so use the other metrics too.


## 3. Install and import libraries

If a package is already installed, the installation command can be skipped.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib joblib

In [ ]:
import re
import html
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42


## 4. Load the IMDB dataset

Download the CSV from the dataset source linked above. Upload `IMDB Dataset.csv` to the same folder as this notebook. In Google Colab, use the file upload panel or run the optional upload cell below.

In [ ]:
# Google Colab users may uncomment and run these two lines:
# from google.colab import files
# uploaded = files.upload()

DATA_PATH = "IMDB Dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumn names:", df.columns.tolist())
print("\\nSentiment counts:")
print(df["sentiment"].value_counts(dropna=False))


## 5. Explore and prepare the data

Check that the required columns exist, remove missing/duplicate records, and standardise the sentiment labels.

In [ ]:
required_columns = {"review", "sentiment"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

df = df[["review", "sentiment"]].dropna().copy()
df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()
df = df[df["sentiment"].isin(["positive", "negative"])]
df = df.drop_duplicates(subset=["review"]).reset_index(drop=True)

print("Clean dataset shape:", df.shape)
display(df["sentiment"].value_counts().to_frame("count"))


## 6. Clean the review text

The cleaning function removes HTML tags, decodes HTML entities, converts text to lowercase, and removes most punctuation/numeric characters. It deliberately keeps negation words such as `not` and `never`.

**Important:** Avoid removing every short/common word automatically. In sentiment analysis, words such as `not`, `no`, and `never` carry important meaning.

In [ ]:
def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)       # remove HTML tags
    text = text.lower()
    text = re.sub(r"[^a-z\\s']", " ", text)   # keep letters and apostrophes
    text = re.sub(r"\\s+", " ", text).strip()
    return text

df["clean_review"] = df["review"].apply(clean_text)

print("Original review:")
print(df.loc[0, "review"][:500])
print("\\nCleaned review:")
print(df.loc[0, "clean_review"][:500])


## 7. Encode labels and split the data

The model uses numeric labels:
- Positive = `1`
- Negative = `0`

We reserve 20% of the data for final testing. `stratify=y` preserves the class proportions in both sets.

In [ ]:
X = df["clean_review"]
y = df["sentiment"].map({"negative": 0, "positive": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))
print("\\nTraining label proportions:")
print(y_train.value_counts(normalize=True))


## 8. Build the TF-IDF + Logistic Regression pipeline

A scikit-learn `Pipeline` combines feature extraction and classification. This keeps preprocessing consistent and makes it easier to save and reuse the complete model.

- `max_features=20000`: limits the vocabulary size.
- `ngram_range=(1, 2)`: uses single words and two-word phrases.
- `min_df=2`: ignores terms appearing in fewer than two training reviews.
- `sublinear_tf=True`: uses a smoothed term-frequency scaling.


In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
print("Training completed.")


## 9. Evaluate the model on test data

The test set was not used to fit the vectorizer or classifier. Therefore, these metrics estimate how the trained pipeline performs on unseen reviews.

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("\\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Negative", "Positive"],
    digits=4
))


## 10. Confusion matrix

For labels ordered as Negative (`0`) then Positive (`1`):
- Top-left: correctly predicted negative reviews (true negatives).
- Top-right: negative reviews incorrectly predicted positive (false positives).
- Bottom-left: positive reviews incorrectly predicted negative (false negatives).
- Bottom-right: correctly predicted positive reviews (true positives).


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)
disp.plot(values_format="d")
plt.title("Movie Review Sentiment - Confusion Matrix")
plt.show()


## 11. Predict sentiment for a new review

The prediction function accepts raw text. It applies the same cleaning function and then sends the cleaned text through the trained pipeline.

In [ ]:
def predict_sentiment(review):
    cleaned_review = clean_text(review)
    prediction = int(model.predict([cleaned_review])[0])
    probabilities = model.predict_proba([cleaned_review])[0]

    label = "Positive" if prediction == 1 else "Negative"
    confidence = float(probabilities[prediction])

    return {
        "review": review,
        "predicted_sentiment": label,
        "confidence": round(confidence, 4)
    }

sample_reviews = [
    "This movie was fantastic. The acting and story were excellent!",
    "The film was boring, poorly written, and a complete waste of time.",
    "I did not expect much, but I really enjoyed this movie."
]

for review in sample_reviews:
    print(predict_sentiment(review))


## 12. Optional: compare other machine-learning algorithms

A useful extension is to compare Logistic Regression with Multinomial Naive Bayes and Linear Support Vector Machine (SVM). All models use the same train/test split and TF-IDF settings.

The comparison below reports test accuracy. For a fairer comparison, also inspect precision, recall, and F1-score.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE)
}

comparison_rows = []

# Fit TF-IDF only on training data, then reuse the same representation.
comparison_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_train_features = comparison_vectorizer.fit_transform(X_train)
X_test_features = comparison_vectorizer.transform(X_test)

for name, estimator in candidate_models.items():
    estimator.fit(X_train_features, y_train)
    predictions = estimator.predict(X_test_features)
    comparison_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "Accuracy", ascending=False
)
display(comparison_df)


## 13. Save the trained model

Saving the entire pipeline stores both TF-IDF feature extraction and the classifier. This means a future program can load the pipeline and predict without fitting the vectorizer again.

In [ ]:
joblib.dump(model, "movie_review_sentiment_pipeline.joblib")
print("Saved: movie_review_sentiment_pipeline.joblib")

# Example of loading the saved model:
loaded_model = joblib.load("movie_review_sentiment_pipeline.joblib")
test_review = "A beautifully made film with wonderful performances."
cleaned = clean_text(test_review)
print("Loaded model prediction:", "Positive" if loaded_model.predict([cleaned])[0] == 1 else "Negative")


## 14. Conclusion

This project demonstrates a complete traditional NLP and machine-learning workflow for movie-review sentiment classification. Reviews are cleaned, transformed into TF-IDF features, and classified using Logistic Regression. The model is evaluated on held-out test data and can classify new user-written reviews.

### Limitations
- Sarcasm and humour can be difficult to interpret.
- A review can contain both positive and negative opinions.
- The model learns patterns from its training dataset and may not generalise equally well to every genre, language, or review platform.
- The confidence value is a model probability, not a guarantee that the prediction is correct.

### Future enhancements
- Add a Neutral sentiment class.
- Compare word-level TF-IDF with word embeddings.
- Try transformer models such as BERT.
- Build a simple web interface using Streamlit.
- Analyse sentiment by sentence or aspect (acting, story, music, direction).

## 15. Viva questions

1. **What is NLP?** Natural Language Processing is a field of AI concerned with processing and analysing human language.
2. **What is sentiment analysis?** It identifies the opinion or emotional polarity expressed in text.
3. **Why do we use TF-IDF?** It converts text into weighted numerical features for machine-learning algorithms.
4. **Why retain negation words?** Words such as “not” can reverse the sentiment of a phrase.
5. **What is supervised learning?** Learning from examples that contain input data and known labels.
6. **Why split training and test data?** To evaluate performance on data that was not used to train the model.
7. **What does precision measure?** The proportion of predicted positive examples that are actually positive.
8. **What does recall measure?** The proportion of actual positive examples correctly identified.
9. **What is overfitting?** When a model learns training-specific patterns and performs poorly on new data.
10. **Why use a pipeline?** It keeps feature extraction and model prediction together and helps prevent inconsistent transformations.
